# SAD AR5 — Análise de dados e validação científica

**CT302 · Grupo VI · Escola Naval**

Notebook reprodutível do pipeline analítico: EDA das apreensões, pesos AHP, grelha de risco multi-ameaça, otimização MCLP/frota e validação (backtest + baseline).

| Secção | Conteúdo |
|--------|----------|
| 0 | Setup e caminhos |
| 1 | Análise exploratória (apreensões UNODC) |
| 2 | Pesos AHP das ameaças |
| 3 | Grelha de risco e mapas |
| 4 | Otimização — MCLP, frota 24 h, sensibilidade |
| 5 | Validação — backtest temporal e baseline patrulha |
| 6 | Respostas SAD Q1–Q3 e exportação |

> **Pré-requisitos:** Python 3.10+, dependências em `requirements.txt`.  
> Executar a partir da raiz do repositório ou ajustar `REPO` na célula 0.


In [ ]:
# Instalar dependências (descomente na 1.ª execução):
# %pip install -q -r requirements.txt matplotlib seaborn jupyter

import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)

# Caminho para a raiz do repositório (ajuste se necessário)
REPO = Path.cwd()
if not (REPO / "src" / "config.py").exists():
    REPO = Path.cwd().parent  # se estiver em notebooks/
if not (REPO / "src" / "config.py").exists():
  raise FileNotFoundError("Não encontrei src/config.py — abra o notebook na pasta do projeto")

SRC = REPO / "src"
sys.path.insert(0, str(SRC))
os.chdir(SRC)

FIGDIR = REPO / "resultados" / "figuras"
OUTDIR = REPO / "resultados"
FIGDIR.mkdir(parents=True, exist_ok=True)

print("Repositório:", REPO)
print("A trabalhar em:", os.getcwd())


## 1. Análise exploratória — apreensões de droga

In [ ]:
from dm.eda import gerar as eda_gerar

resumo_eda = eda_gerar()
print("Resumo EDA:", json.dumps(resumo_eda, indent=2, ensure_ascii=False))
from IPython.display import Image, display
display(Image(filename=str(REPO / "resultados/dm/eda_panorama.png")))


## 2. Pesos AHP (Processo de Hierarquia Analítica)

In [ ]:
from dm.ahp_pesos import main as ahp_main

ahp = ahp_main()
print(f"CR={ahp['consistency_ratio']}  consistente={ahp['consistente']}")
print("Pesos AHP:", ahp["pesos_ahp"])
print("Pesos adoptados:", ahp["pesos_adotados"])
display(Image(filename=str(FIGDIR / "24_ahp_pesos.png")))


## 3. Grelha de risco multi-ameaça

In [ ]:
from geo import gerar_procura, bases_proj, COSTA_LONLAT, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX
from risco import calcular_risco
from config import PESOS_AMEACA, LIMIAR_RISCO_OPERACIONAL

XLSX = str(REPO / "dados/fontes/apreensoes_droga_PT.xlsx")
LIMIAR = LIMIAR_RISCO_OPERACIONAL

pts = gerar_procura()
bases = bases_proj()
calcular_risco(pts, XLSX)

lon = [p["lon"] for p in pts]
lat = [p["lat"] for p in pts]
r = [p["risco"] for p in pts]
alto = sum(1 for x in r if x >= LIMIAR)

print(f"Células: {len(pts)} | Alto risco (≥{LIMIAR}): {alto}")
print("Pesos:", PESOS_AMEACA)

fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(lon, lat, c=r, cmap="YlOrRd", s=14, marker="s", vmin=0, vmax=1)
costa_lon = [c[0] for c in COSTA_LONLAT]
costa_lat = [c[1] for c in COSTA_LONLAT]
ax.plot(costa_lon, costa_lat, color="#444", lw=1.5)
ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_title("Índice de risco SAD — Portugal Continental")
plt.colorbar(sc, ax=ax, label="Risco [0–1]")
fig.tight_layout()
fig.savefig(FIGDIR / "01_risco_notebook.png", dpi=140)
plt.show()


## 4. Otimização — MCLP, frota e sensibilidade

In [ ]:
# Pipeline completo de otimização (figuras + resultados.json)
import main as pipeline

pipeline.main()


## 5. Validação científica — backtest e baseline

In [ ]:
import validacao

validacao.main()

with open(OUTDIR / "validacao.json", encoding="utf-8") as f:
    val = json.load(f)

bt = val["backtest_temporal"]
bl = val["baseline_patrulha"]
print("--- Backtest temporal ---")
print(f"  Holdout n={bt['n_holdout']}  acerto limiar={bt['taxa_acerto_limiar']:.1%}")
print(f"  Ganho relativo={bt['ganho_relativo_limiar']}×")
print("--- Baseline patrulha ---")
print(f"  N células={bl['n_celulas_patrulha']}  ganho SAD vs aleatório={bl['ganho_sad_vs_aleatorio']}×")

for fig in ("21_backtest_temporal.png", "22_baseline_patrulha.png"):
    p = FIGDIR / fig
    if p.exists():
        display(Image(filename=str(p)))


## 6. Respostas SAD — Q1, Q2, Q3

In [ ]:
resp = val["resposta_objetivo"]
rows = [
    ("Q1 — Onde patrulhar?", resp["Q1_onde"]["resposta"]),
    ("Q2 — Quantos AR5?", resp["Q2_quantos"]["resposta"]),
    ("Q3 — Onde colocar bases?", resp["Q3_bases"]["resposta"]),
]
pd.DataFrame(rows, columns=["Pergunta", "Resposta"])


## Notas

- Para **regenerar intensidades** a partir dos rasters EMODnet: `cd src && python -m dm.construir_dados_reais` (requer `rasterio`).
- Os JSON em `resultados/` alimentam a **plataforma web** em runtime.
- Classificação marítimo/terrestre e clustering: `python -m dm.main_dm`.
